In [ ]:
# Paths come from `paths.py`, never from a literal relative to some checkout.
# That module is the only place that knows where the tree lives, and it names
# the environment variables that override each location (FSCORE_DB above all —
# fscore.db is ~1.7 GB and is not kept in the repository).
# Run this notebook from its own directory, src/fscore_vietnam.
import sys, pathlib

HERE = pathlib.Path.cwd()
assert (HERE / "paths.py").exists(), f"run from src/fscore_vietnam (cwd={HERE})"
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from paths import DATA, RESULTS, DB, ensure_dirs, require_db
ensure_dirs()

# Price matching and finalize

The last stage of the preprocessing pipeline. Three panels come in, one comes out:

| panel | notebook | what it carries |
|---|---|---|
| `f_score_panel.csv` | `f_score_calculation` | 9 raw signals, 9 indicators, `f_score` |
| `book_to_market_panel.csv` | `book_to_market_calculation` | book equity, shares, 31 Dec close, `bm` |
| `trade_turnover_panel.csv` | `trade_turnover_filter` | June window volumes, `turnover`, `tradeable` |

All three key on `(symbol, period)`, where `period` is the **fiscal year**. What they do not
carry is the price you would actually transact at, and that price does not live in the
fiscal year: the portfolio is decided on **30 June of `period` + 1**, once the accounting for
`period` has been published, and it is held over the year that opens the next day.

    formation date  = 30 June of year F   (the decision: F-score, B/M, liquidity)
    matched to      = period F - 1
    holding period  = 1 July F  ->  30 June F + HOLDING_YEARS
    entry price     = the first session on or after 1 July F
    exit price      = the last session on or before 30 June F + HOLDING_YEARS

**The decision date and the entry date are different days, on purpose.** Everything the
screen reads is known by the close of 30 June — the June turnover window ends there, the
accounting landed in March, the book-to-market is measured against the 31 December close —
and the position opens at the next session the market offers. Buying at the 30 June close
would transact at a price printed on the same day as the decision that selected it; opening
on 1 July cannot, whatever the screen saw.

**Each end reaches into the holding year, never out of it.** The entry takes the first
session *on or after* 1 July and the exit the last session *on or before* 30 June, so the
two prices bracket the holding year from the inside and the whole return is earned within
it. Neither reach costs more than a weekend on a name the strategy could actually buy:
11,426 of the 11,444 tradeable rows are priced within two days of 1 July, and 8,670 of the
8,769 finished holdings exit within three days of 30 June.

In [1]:
import sqlite3
from contextlib import closing

import numpy as np
import pandas as pd

from statement_fields import SHARE_COLS


# The formation date — the day the screen is run — is 30 June, and it lives in
# trade_turnover_filter.ipynb as FORMATION_MMDD. Nothing is transacted on it. The holding
# year opens the next session and closes on the following 30 June.
ENTRY_MMDD = "07-01"       # first session on or after this date opens the position
EXIT_MMDD = "06-30"        # last session on or before this date closes it
FORMATION_LAG_YEARS = 1    # June of year F screens fiscal period F - 1

HOLDING_YEARS = 1          # 1 July F -> 30 June F + 1
MAX_ENTRY_LEAD_DAYS = 30   # beyond this the entry print is not an entry-day price
MAX_EXIT_LAG_DAYS = 30     # beyond this the exit print is not an exit-day price
DELISTING_RETURN = None    # None = exit at the last traded price; or a number, e.g. -0.30

CUTOFF = 8                 # F-score cutoff, used only for the previews at the end

## Inputs

Read as written, no re-derivation. If any of the three is stale the index check below says
so immediately rather than letting a silent left-join drop rows.

In [2]:
f_score_panel = pd.read_csv(f"{RESULTS}/f_score_panel.csv").set_index(["symbol", "period"])
bm_panel = pd.read_csv(f"{RESULTS}/book_to_market_panel.csv").set_index(["symbol", "period"])
turnover_panel = pd.read_csv(f"{RESULTS}/trade_turnover_panel.csv").set_index(["symbol", "period"])

pd.DataFrame({
    "rows": [len(f_score_panel), len(bm_panel), len(turnover_panel)],
    "columns": [f_score_panel.shape[1], bm_panel.shape[1], turnover_panel.shape[1]],
}, index=["f_score", "book_to_market", "trade_turnover"])

,rows,columns
f_score,21233,20
book_to_market,21233,13
trade_turnover,21233,22


In [3]:
# All three are built from the same accounting-clean universe, so their indexes must be
# identical. A difference means one notebook was run against a different vintage of
# f_score_fields_extract_corrected.csv, and every count downstream would be off.
index_check = pd.Series({
    "f_score == bm": f_score_panel.index.equals(bm_panel.index),
    "bm == turnover": bm_panel.index.equals(turnover_panel.index),
    "rows in f_score only": len(f_score_panel.index.difference(bm_panel.index)),
    "rows in bm only": len(bm_panel.index.difference(f_score_panel.index)),
    "rows in turnover only": len(turnover_panel.index.difference(bm_panel.index)),
})
assert index_check["f_score == bm"] and index_check["bm == turnover"], \
    "panels disagree — re-run the notebook that is behind before finalising"
index_check

f_score == bm            True
bm == turnover           True
rows in f_score only        0
rows in bm only             0
rows in turnover only       0
dtype: object

## The join

Six columns are computed in both the BM and the turnover notebook: `has_treasury`, and the
five the share count produces. Both notebooks get them from the same `paid_in_capital` and
the same vendor count through the same `statement_fields.share_count()`, so they must agree
to the share — one divides price by `shares`, the other divides volume by it, and a silent
disagreement would mean the value screen and the liquidity screen were ranking different
companies. Checked here, then kept once.

The BM notebook's `close` is the **31 December** close, the price the ratio was measured
against. The formation price added below is a different date and a different purpose, so
the December one is renamed on the way in rather than left to collide.

In [4]:
DUPLICATED = SHARE_COLS + ["has_treasury"]

for col in DUPLICATED:
    a, b = bm_panel[col], turnover_panel[col]
    agree = (np.allclose(a.fillna(-1), b.fillna(-1))
             if pd.api.types.is_numeric_dtype(a) else a.equals(b))
    assert agree, f"{col} disagrees between the BM and turnover panels — re-run both"

panel = (
    f_score_panel
    .join(bm_panel.rename(columns={"close": "close_dec",
                                   "close_date": "close_dec_date",
                                   "price_stale": "close_dec_stale"}))
    .join(turnover_panel.drop(columns=DUPLICATED))
)

print(f"{panel.shape[0]} rows x {panel.shape[1]} columns")
panel.columns.tolist()

21233 rows x 49 columns


['raw_roa',
 'raw_cfoa',
 'raw_d_roa',
 'raw_accrual',
 'raw_d_lever',
 'raw_d_liquid',
 'raw_eq_offer',
 'raw_d_margin',
 'raw_d_turn',
 'f_roa',
 'f_cfoa',
 'f_d_roa',
 'f_accrual',
 'f_d_lever',
 'f_d_liquid',
 'f_eq_offer',
 'f_d_margin',
 'f_d_turn',
 'signals_available',
 'f_score',
 'book_equity',
 'shares_issued',
 'shares_out',
 'shares_gap',
 'shares_source',
 'shares',
 'close_dec',
 'market_equity',
 'bm',
 'bm_shares_issued',
 'close_dec_date',
 'close_dec_stale',
 'has_treasury',
 'formation_year',
 'market_sessions',
 'bars',
 'traded_sessions',
 'traded_pct',
 'deal_volume',
 'total_volume',
 'deal_value_vnd',
 'last_bar',
 'turnover',
 'turnover_issued',
 'turnover_total',
 'putthrough_pct',
 'adtv_vnd',
 'min_turnover',
 'tradeable']

## The entry price

One row per `(symbol, formation_year)`: the first session on or after 1 July of that year.
`price_close * unit` is the **raw** quote, the same convention the BM notebook uses.

`adj_ratio` rides along, and `close_adj = close_raw / adj_ratio` is the split- and
dividend-adjusted series — FireAnt's ratio adjusts for cash dividends too, so a return
computed from `close_adj` is a total return. It is cumulative to the crawl date, which makes
it internally consistent across this whole panel but not comparable to a series fetched
later. Use `close_raw` for anything that has to be the price on the day, `close_adj` for
anything that spans two dates.

**The search stops at the year end.** The window is partitioned on the calendar year, so a
symbol whose last print of year F falls before 1 July gets no entry price and therefore no
position — which is the right answer: it could not have been bought. The old rule, reaching
*backwards* from 30 June, would have priced that row off a stale print from a stock that had
already stopped trading.

Nothing is dropped for staleness here: `entry_lead_days` and `price_traded` are exposed and
`tradeable` is left to be the single gate. It already is one — of the 11,444 rows the
liquidity filter passes, 11,426 are priced within two days of 1 July and exactly one waits
longer than `MAX_ENTRY_LEAD_DAYS`. Twelve tradeable rows never print again inside their
formation year and so carry no entry price at all; they leave the investable universe by
losing their price, which in this market is the same statement as "could not be bought".

In [5]:
AS_OF_PRICE_SQL = f"""
select symbol,
       formation_year,
       date                as price_date,
       price_close * unit  as close_raw,
       adj_ratio,
       total_volume > 0    as price_traded,
       cast(julianday(date)
            - julianday(substr(date, 1, 4) || '-{ENTRY_MMDD}') as integer) as entry_lead_days
from (
    select symbol,
           cast(substr(date, 1, 4) as integer) as formation_year,
           date, price_close, unit, adj_ratio, total_volume,
           row_number() over (partition by symbol, substr(date, 1, 4)
                              order by date asc) as rn
    from fireant_prices
    where unit = 1000
      and price_close > 0
      and date >= substr(date, 1, 4) || '-{ENTRY_MMDD}'
)
where rn = 1
"""

with closing(sqlite3.connect(DB)) as conn:
    formation_price = pd.read_sql(AS_OF_PRICE_SQL, conn)

formation_price["close_adj"] = formation_price["close_raw"] / formation_price["adj_ratio"]
formation_price["price_traded"] = formation_price["price_traded"].astype(bool)
formation_price["period"] = formation_price["formation_year"] - FORMATION_LAG_YEARS
formation_price = (formation_price
                   .drop(columns="formation_year")
                   .set_index(["symbol", "period"])
                   .sort_index())
formation_price

price_date  close_raw  adj_ratio  price_traded  \
symbol period                                                   
A32    2017    2018-10-23    25900.0   1.915475         False   
       2018    2019-07-01    22500.0   1.777279         False   
       2019    2020-07-01    26300.0   1.642451         False   
       2020    2021-07-01    32000.0   1.577788         False   
       2021    2022-07-01    35200.0   1.497749          True   
...                   ...        ...        ...           ...   
YTC    2021    2022-07-01    65500.0   2.092963         False   
       2022    2023-07-03    59000.0   2.092963         False   
       2023    2024-07-01    36000.0   1.010567         False   
       2024    2025-07-01    34000.0   1.000000         False   
       2025    2026-07-01    25500.0   1.000000         False   

               entry_lead_days     close_adj  
symbol period                                 
A32    2017                114  13521.447847  
       2018                  0  12659.801435  
       2019                  0  16012.654319  
       2020                  0  20281.564282  
       2021                  0  23501.932498  
...                        ...           ...  
YTC    2021                  0  31295.347018  
       2022                  2  28189.701894  
       2023                  0  35623.576422  
       2024                  0  34000.000000  
       2025                  0  25500.000000  

[21346 rows x 6 columns]

In [6]:
# How far the matched print sits past 1 July. Anything beyond a long weekend is a stock that
# was not printing when the holding year opened, not a calendar effect.
lead = formation_price["entry_lead_days"]

pd.Series({
    "symbol-years priced": len(formation_price),
    "exactly on the entry date": int(lead.eq(0).sum()),
    "  within 2 days (weekend)": int(lead.le(2).sum()),
    f"  3 to {MAX_ENTRY_LEAD_DAYS} days": int(lead.between(3, MAX_ENTRY_LEAD_DAYS).sum()),
    f"  over {MAX_ENTRY_LEAD_DAYS} days": int(lead.gt(MAX_ENTRY_LEAD_DAYS).sum()),
    "longest wait, days": int(lead.max()),
    "priced off a session with no trade": int((~formation_price["price_traded"]).sum()),
})

symbol-years priced                   21346
exactly on the entry date             15542
  within 2 days (weekend)             20495
  3 to 30 days                          158
  over 30 days                          693
longest wait, days                      183
priced off a session with no trade     8323
dtype: int64

In [7]:
panel = panel.join(formation_price[["price_date", "entry_lead_days", "price_traded",
                                    "close_raw", "adj_ratio", "close_adj"]])

# The turnover notebook's `last_bar` is the last bar inside the 30-day screening window,
# which ends 30 June; the entry print is the first bar of the holding year. The two must
# therefore never be the same session, and the entry must always come after — that is the
# check that the screen and the trade are on opposite sides of the formation date.
both_bars = panel["last_bar"].notna() & panel["price_date"].notna()
print(f"rows where both notebooks found a bar: {int(both_bars.sum())}, "
      f"entry not strictly after the screening window: "
      f"{int((panel.loc[both_bars, 'price_date'] <= panel.loc[both_bars, 'last_bar']).sum())}")

panel[["f_score", "bm", "turnover", "tradeable", "price_date", "close_raw", "close_adj"]]

rows where both notebooks found a bar: 18494, entry not strictly after the screening window: 0


f_score        bm  turnover tradeable  price_date  close_raw  \
symbol period                                                                 
A32    2017        NaN       NaN       NaN       NaN  2018-10-23    25900.0   
       2018        NaN  0.977925  0.000544     False  2019-07-01    22500.0   
       2019        8.0  1.174454  0.000779     False  2020-07-01    26300.0   
       2020        6.0  1.032493  0.007485      True  2021-07-01    32000.0   
       2021        5.0  1.100893  0.001441     False  2022-07-01    35200.0   
...                ...       ...       ...       ...         ...        ...   
YTC    2021        6.0  0.128374  0.000065     False  2022-07-01    65500.0   
       2022        9.0  0.095641  0.000032     False  2023-07-03    59000.0   
       2023        6.0  0.093050  0.002662      True  2024-07-01    36000.0   
       2024        8.0  0.470456  0.000346      True  2025-07-01    34000.0   
       2025        4.0  0.676676  0.000021     False  2026-07-01    25500.0   

                  close_adj  
symbol period                
A32    2017    13521.447847  
       2018    12659.801435  
       2019    16012.654319  
       2020    20281.564282  
       2021    23501.932498  
...                     ...  
YTC    2021    31295.347018  
       2022    28189.701894  
       2023    35623.576422  
       2024    34000.000000  
       2025    25500.000000  

[21233 rows x 7 columns]

## Coverage of the joined panel

Each block can be missing for its own reason, and they do not overlap neatly: an F-score
needs three consecutive clean years, a BM needs a December close and positive book equity, a
turnover needs bars in the June window. The row that matters is the last one — everything
present, and tradeable.

In [8]:
have = pd.DataFrame({
    "f_score": panel["f_score"].notna(),
    "bm": panel["bm"].notna(),
    "turnover": panel["turnover"].notna(),
    "price": panel["close_raw"].notna(),
})
complete = have.all(axis=1)
investable = complete & panel["tradeable"].fillna(False)

pd.Series({
    "panel rows": len(panel),
    "with f_score": int(have["f_score"].sum()),
    "with bm": int(have["bm"].sum()),
    "with turnover": int(have["turnover"].sum()),
    "with a formation price": int(have["price"].sum()),
    "complete (all four)": int(complete.sum()),
    "complete and tradeable": int(investable.sum()),
    f"  and f_score >= {CUTOFF}": int((investable & panel["f_score"].ge(CUTOFF)).sum()),
})

panel rows                21233
with f_score              16564
with bm                   17414
with turnover             18542
with a formation price    19064
complete (all four)       14778
complete and tradeable     9482
  and f_score >= 8         1116
dtype: int64

In [9]:
# The same by fiscal period, with the formation year each row is priced at. `investable` is
# the universe a backtest actually ranks; `selected` is what the strategy buys out of it.
pd.DataFrame({
    "formation": panel.groupby(level="period")["formation_year"].first(),
    "rows": panel.groupby(level="period").size(),
    "complete": complete.groupby(level="period").sum(),
    "investable": investable.groupby(level="period").sum(),
    "selected": (investable & panel["f_score"].ge(CUTOFF)).groupby(level="period").sum(),
    "median_bm": panel.loc[investable, "bm"].groupby(level="period").median().round(3),
    "median_turnover": panel.loc[investable, "turnover"].groupby(level="period").median().round(4),
})

,formation,rows,complete,investable,selected,median_bm,median_turnover
period,,,,,,,
2009,2010.0,525,0,0,0,NaN,NaN
2010,2011.0,751,0,0,0,NaN,NaN
2011,2012.0,875,408,329,26,1.896,0.0205
2012,2013.0,984,548,416,29,1.753,0.0175
2013,2014.0,1060,607,429,51,1.513,0.0154
2014,2015.0,1111,648,473,69,1.140,0.0198
2015,2016.0,1255,715,515,48,1.096,0.0251
2016,2017.0,1369,816,553,65,1.066,0.0183
2017,2018.0,1482,1038,540,59,0.979,0.0095


In [10]:
# What each screen costs, in order, on the rows that have an F-score at all. Read down: this
# is the funnel from "scored" to "bought".
scored = have["f_score"]
funnel = pd.Series({
    "scored": int(scored.sum()),
    "+ has bm": int((scored & have["bm"]).sum()),
    "+ has a formation price": int((scored & have["bm"] & have["price"]).sum()),
    "+ tradeable": int(investable.sum()),
    f"+ f_score >= {CUTOFF}": int((investable & panel["f_score"].ge(CUTOFF)).sum()),
})
funnel.to_frame("rows").assign(pct_of_scored=lambda d: (100 * d["rows"] / funnel.iloc[0]).round(1))

,rows,pct_of_scored
scored,16564,100.0
+ has bm,14873,89.8
+ has a formation price,14782,89.2
+ tradeable,9482,57.2
+ f_score >= 8,1116,6.7


## Exit price and forward return

The position opens at `close_adj` on the first session on or after 1 July F and closes at
`close_adj` on the last session on or before 30 June F + `HOLDING_YEARS`. Both ends
adjusted, always — a raw entry against an adjusted exit, or either end raw, books every
dividend and every bonus issue as a capital loss.

**The exit is looked up as-of, not by shifting the panel.** The panel's rows are gated by
the accounting checks, so a firm whose *next* year's statements failed them loses its row —
while the stock kept trading and the return exists. Shifting drops those, and they are not a
random sample: on this panel they run about 7pp below the rest, so dropping them flatters
the result.

**`exit_lag_days` is the whole delisting story.** There is no delisting flag anywhere in
`fireant_prices`; the only observable is that a symbol stops printing. The gap between the
exit date and the last bar before it is the evidence, and it splits three ways: a long
weekend, a thin stretch, or a stock that left. `exit_permanent` marks the last case — no
print for over `MAX_EXIT_LAG_DAYS` before the exit *and* none after it either. A stock that
merely paused and came back is not marked, because it did not leave.

**`DELISTING_RETURN` is left at `None` on purpose.** The default exits at the last traded
price, which invents nothing, but it is optimistic: some of those positions come out at
exactly the entry price, and a stock thrown off the exchange rarely leaves a holder whole.
The sensitivity table below is the answer to that, rather than a haircut picked to look
defensible. `-1.0` would be the wrong default in this market: a ticker that moves from HOSE
to UPCoM, merges, or simply leaves the 1,830-symbol crawl universe looks identical to a
bankruptcy, and booking it at -100% fabricates a loss that never happened.

In [11]:
# The as-of lookup only needs month-end bars, because the exit date is the last calendar day
# of a month: the last bar on or before it is the last bar of that month, or of the most
# recent earlier month that has one. If EXIT_MMDD ever moves off a month end this
# reduction stops being valid and the merge has to run on daily bars.
MONTH_END_SQL = """
select symbol, date, price_close * unit / adj_ratio as close_adj
from (
    select symbol, date, price_close, unit, adj_ratio,
           row_number() over (partition by symbol, substr(date, 1, 7)
                              order by date desc) as rn
    from fireant_prices
    where unit = 1000 and price_close > 0
)
where rn = 1
"""

with closing(sqlite3.connect(DB)) as conn:
    month_end = pd.read_sql(MONTH_END_SQL, conn, parse_dates=["date"])

month_end = month_end.sort_values("date")
data_end = month_end["date"].max()
last_print = month_end.groupby("symbol")["date"].max()

print(f"{len(month_end)} month-end bars, {month_end['symbol'].nunique()} symbols, "
      f"price data ends {data_end.date()}")

241632 month-end bars, 1824 symbols, price data ends 2026-08-07


In [12]:
period = panel.index.get_level_values("period")
exit_year = period + FORMATION_LAG_YEARS + HOLDING_YEARS

targets = pd.DataFrame({
    "symbol": panel.index.get_level_values("symbol"),
    "period": period,
    "exit_date": pd.to_datetime(exit_year.astype(str) + f"-{EXIT_MMDD}"),
}).sort_values("exit_date")

exit_px = (pd.merge_asof(targets, month_end, left_on="exit_date", right_on="date",
                         by="symbol", direction="backward")
           .set_index(["symbol", "period"])
           .reindex(panel.index))

panel["exit_date"] = exit_px["exit_date"]
panel["exit_price_date"] = exit_px["date"]
panel["close_adj_exit"] = exit_px["close_adj"]
panel["exit_lag_days"] = (exit_px["exit_date"] - exit_px["date"]).dt.days

# The holding period has to have finished. Left alone, merge_asof prices a 2027 exit off the
# last bar in the table, which is not an exit — it is an open position, and counting it would
# put a partial year into the cross-section as if it were a full one.
panel["holding_complete"] = panel["exit_date"].le(data_end)

# Permanently gone: nothing printed for MAX_EXIT_LAG_DAYS before the exit, and nothing after
# it either. The second half is what separates a delisting from a long suspension.
resumed = pd.Series(last_print.reindex(panel.index.get_level_values("symbol")).to_numpy(),
                    index=panel.index).gt(panel["exit_date"])
panel["exit_permanent"] = (panel["holding_complete"]
                           & panel["exit_lag_days"].gt(MAX_EXIT_LAG_DAYS)
                           & ~resumed)

panel[["price_date", "close_adj", "exit_date", "exit_price_date", "close_adj_exit",
       "exit_lag_days", "holding_complete", "exit_permanent"]]

price_date     close_adj  exit_date exit_price_date  \
symbol period                                                        
A32    2017    2018-10-23  13521.447847 2019-06-30      2019-06-28   
       2018    2019-07-01  12659.801435 2020-06-30      2020-06-30   
       2019    2020-07-01  16012.654319 2021-06-30      2021-06-30   
       2020    2021-07-01  20281.564282 2022-06-30      2022-06-30   
       2021    2022-07-01  23501.932498 2023-06-30      2023-06-30   
...                   ...           ...        ...             ...   
YTC    2021    2022-07-01  31295.347018 2023-06-30      2023-06-30   
       2022    2023-07-03  28189.701894 2024-06-30      2024-06-28   
       2023    2024-07-01  35623.576422 2025-06-30      2025-06-30   
       2024    2025-07-01  34000.000000 2026-06-30      2026-06-30   
       2025    2026-07-01  25500.000000 2027-06-30      2026-08-07   

               close_adj_exit  exit_lag_days  holding_complete  exit_permanent  
symbol period                                                                   
A32    2017      12659.801435            2.0              True           False  
       2018      16012.654319            0.0              True           False  
       2019      20281.564282            0.0              True           False  
       2020      23568.699351            0.0              True           False  
       2021      22148.190511            0.0              True           False  
...                       ...            ...               ...             ...  
YTC    2021      28189.701894            0.0              True           False  
       2022      35821.485180            2.0              True           False  
       2023      34000.000000            0.0              True           False  
       2024      25500.000000            0.0              True           False  
       2025      21700.000000          327.0             False           False  

[21233 rows x 8 columns]

In [13]:
# How the exit prints sit against the exit date, on finished holding periods only.
finished = investable & panel["holding_complete"]
exit_lag = panel.loc[finished, "exit_lag_days"]

pd.Series({
    "investable, holding period finished": int(finished.sum()),
    "  no exit price at all": int((finished & panel["close_adj_exit"].isna()).sum()),
    "exit lag <= 3 days (weekend)": int(exit_lag.le(3).sum()),
    f"  4 to {MAX_EXIT_LAG_DAYS} days": int(exit_lag.between(4, MAX_EXIT_LAG_DAYS).sum()),
    f"  over {MAX_EXIT_LAG_DAYS} days": int(exit_lag.gt(MAX_EXIT_LAG_DAYS).sum()),
    "    traded again later (paused)": int((finished & exit_lag.gt(MAX_EXIT_LAG_DAYS)
                                            & ~panel["exit_permanent"]).sum()),
    "    never traded again (left)": int((finished & panel["exit_permanent"]).sum()),
    "investable, still open (no exit yet)": int((investable & ~panel["holding_complete"]).sum()),
})

investable, holding period finished     8769
  no exit price at all                     0
exit lag <= 3 days (weekend)            8670
  4 to 30 days                            21
  over 30 days                            78
    traded again later (paused)           17
    never traded again (left)             61
investable, still open (no exit yet)     713
dtype: int64

In [14]:
gross_return = panel["close_adj_exit"] / panel["close_adj"] - 1
panel["fwd_return_1y"] = gross_return.where(panel["holding_complete"])

if DELISTING_RETURN is not None:
    panel.loc[panel["exit_permanent"], "fwd_return_1y"] = DELISTING_RETURN

selected = finished & panel["f_score"].ge(CUTOFF)
pd.DataFrame({
    "n": [int(finished.sum()), int(selected.sum())],
    "median": [panel.loc[finished, "fwd_return_1y"].median(),
               panel.loc[selected, "fwd_return_1y"].median()],
    "mean": [panel.loc[finished, "fwd_return_1y"].mean(),
             panel.loc[selected, "fwd_return_1y"].mean()],
}, index=["investable", f"selected (F >= {CUTOFF})"]).round(4)

,n,median,mean
investable,8769,0.0392,0.1594
selected (F >= 8),1015,0.0838,0.1910


In [15]:
# What the delisting policy is worth. It bites on the `exit_permanent` rows only, and those
# are a fraction of a percent of the panel — the point of the table is that the conclusion
# does not depend on which number goes into DELISTING_RETURN, so none of them has to be
# argued for. Report it and the question is closed.
def with_policy(delisting_return):
    r = gross_return.where(panel["holding_complete"])
    if delisting_return is not None:
        r = r.mask(panel["exit_permanent"], delisting_return)
    return r


pd.DataFrame([{
    "policy": "last traded price" if dr is None else f"{dr:.0%}",
    "investable median": round(with_policy(dr)[finished].median(), 4),
    "investable mean": round(with_policy(dr)[finished].mean(), 4),
    f"F>={CUTOFF} median": round(with_policy(dr)[selected].median(), 4),
    f"F>={CUTOFF} mean": round(with_policy(dr)[selected].mean(), 4),
} for dr in [None, -0.30, -0.50, -1.00]]).set_index("policy")

,investable median,investable mean,F>=8 median,F>=8 mean
policy,,,,
last traded price,0.0392,0.1594,0.0838,0.1910
-30%,0.0377,0.1577,0.0765,0.1868
-50%,0.0377,0.1563,0.0765,0.1846
-100%,0.0377,0.1528,0.0765,0.1792


In [16]:
# Forward return by fiscal period. The last period is empty by construction: its holding year
# has not finished inside the price data.
pd.DataFrame({
    "investable": finished.groupby(level="period").sum(),
    "inv_median": panel.loc[finished, "fwd_return_1y"].groupby(level="period").median().round(4),
    "selected": selected.groupby(level="period").sum(),
    "sel_median": panel.loc[selected, "fwd_return_1y"].groupby(level="period").median().round(4),
    "gone": (finished & panel["exit_permanent"]).groupby(level="period").sum(),
})

,investable,inv_median,selected,sel_median,gone
period,,,,,
2009,0,NaN,0,NaN,0
2010,0,NaN,0,NaN,0
2011,329,0.0000,26,0.2827,3
2012,416,0.3312,29,0.4058,7
2013,429,0.1542,51,0.2260,4
2014,473,0.0909,69,0.1364,1
2015,515,0.0789,48,0.0837,0
2016,553,-0.0708,65,-0.0303,3
2017,540,0.0563,59,0.0555,1


In [17]:
# Spot check: one firm, four years, end to end.
panel.loc["HPG", ["f_score", "bm", "turnover", "tradeable", "price_date", "close_adj",
                  "exit_price_date", "close_adj_exit", "fwd_return_1y"]].tail(4)

,f_score,bm,turnover,tradeable,price_date,close_adj,exit_price_date,close_adj_exit,fwd_return_1y
period,,,,,,,,,
2022,5.0,0.917273,0.096770,True,2023-07-03,17720.126020,2024-06-28,21054.485611,0.188168
2023,4.0,0.632344,0.078287,True,2024-07-01,21091.684349,2025-06-30,20265.872369,-0.039153
2024,5.0,0.670870,0.093760,True,2025-07-01,20221.233884,2026-06-30,23300.000000,0.152254
2025,6.0,0.637514,0.048001,True,2026-07-01,23450.000000,2026-08-07,22000.000000,NaN


### Export

`final_panel.csv`, keyed on `(symbol, period)` — one row per firm-year, carrying the score,
the valuation, the liquidity verdict, the price it would be bought at, the price it would be
sold at, and the return between them. Downstream this is the only file the backtest has to
read.

Read `fwd_return_1y` together with `holding_complete` and `exit_permanent`: the first says
whether the holding year finished at all, the second whether the exit price is a real exit
or the last print of a stock that left.

In [18]:
panel.to_csv(f"{RESULTS}/final_panel.csv")
panel.shape

(21233, 62)